# Load Data and Model

In [1]:
import numpy as np
import xgboost as xgb

import numpy as np
import xgboost as xgb
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model

# Loading Data
data_path = "Atom_KSD_Data"

X_train = np.load(f"{data_path}/X_train.npy")
y_train = np.load(f"{data_path}/y_train.npy")

X_val = np.load(f"{data_path}/X_val.npy")
y_val = np.load(f"{data_path}/y_val.npy")

X_test = np.load(f"{data_path}/X_test.npy")
y_test = np.load(f"{data_path}/y_test.npy")

print("Shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# ============================================================
# Load models
# ============================================================

# XGBoost model
print("Loading XGBoost Model ")
xgboost_model = xgb.Booster()
xgboost_model.load_model(
    "saved_models/xgboost_model_fault-69_KSD-70.json"
)
print("Completed")

# ANN model
print("Loading MLP Model")
mlp_model = load_model(
    "saved_models/mlp_model_fault-69_KSD-70.keras"
)
print("Completed")


2026-01-17 14:36:27.878634: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-17 14:36:27.920732: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-17 14:36:28.994693: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Shapes:
X_train: (753687, 70)
y_train: (753687,)
X_val: (188370, 70)
y_val: (188370,)
X_test: (314088, 70)
y_test: (314088,)
Loading XGBoost Model 
Completed
Loading MLP Model
Completed


# XGBoost Accuracy Computation

In [2]:
# ======================================== XGBoost accuracy calculator ====================================
import xgboost as xgb
import numpy as np
import time
import os
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# -------------------------------------------------
# Ensure float32 for faster inference
# -------------------------------------------------
X_train = X_train.astype(np.float32)
X_val   = X_val.astype(np.float32)
X_test  = X_test.astype(np.float32)


# Use all CPU cores
xgboost_model.set_param({"nthread": os.cpu_count()})

# -------------------------------------------------
# Fast batched evaluator with progress bar
# -------------------------------------------------
def eval_split_progress(name, X, y, batch_size=50000):
    print("\n" + "=" * 60)
    print(f"Evaluating {name}")
    print("=" * 60)

    N = len(X)
    preds = []

    start = time.perf_counter()

    for i in tqdm(range(0, N, batch_size), desc=f"{name} Predict", unit="batch"):
        X_batch = X[i:i+batch_size]
        dmat = xgb.DMatrix(X_batch)

        # Prediction (fast C++)
        y_prob = xgboost_model.predict(dmat)
        preds.append(np.argmax(y_prob, axis=1))

    y_pred = np.concatenate(preds)
    elapsed = time.perf_counter() - start

    acc = accuracy_score(y, y_pred)

    print(f"{name} Accuracy : {acc:.6f}")
    print(f"{name} Time     : {elapsed:.4f} sec")

    return acc

# -------------------------------------------------
# Run evaluations
# -------------------------------------------------
train_acc = eval_split_progress("Train", X_train, y_train)
val_acc   = eval_split_progress("Validation", X_val, y_val)
test_acc  = eval_split_progress("Test", X_test, y_test)

# -------------------------------------------------
# Final summary
# -------------------------------------------------
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Train Accuracy      : {train_acc:.6f}")
print(f"Validation Accuracy : {val_acc:.6f}")
print(f"Test Accuracy       : {test_acc:.6f}")



Evaluating Train


Train Predict: 100%|██████████| 16/16 [01:26<00:00,  5.43s/batch]


Train Accuracy : 0.930954
Train Time     : 86.9197 sec

Evaluating Validation


Validation Predict: 100%|██████████| 4/4 [00:20<00:00,  5.19s/batch]


Validation Accuracy : 0.842146
Validation Time     : 20.7516 sec

Evaluating Test


Test Predict: 100%|██████████| 7/7 [00:34<00:00,  4.98s/batch]

Test Accuracy : 0.843646
Test Time     : 34.8324 sec

FINAL SUMMARY
Train Accuracy      : 0.930954
Validation Accuracy : 0.842146
Test Accuracy       : 0.843646


# MLP Accuracy Computation

In [3]:
# ==================================================== MLP Accuracy computation =========================================

import tensorflow as tf
import time
import numpy as np
import pandas as pd
import os

import numpy as np


mlp_model.summary()

# -------------------------------------------------
# Helper: evaluate with timing + progress
# -------------------------------------------------
def eval_split(name, X, y, batch_size=4096):
    print("\n" + "=" * 60)
    print(f"Evaluating {name}")
    print("=" * 60)

    start = time.time()
    loss, acc = mlp_model.evaluate(
        X, y,
        batch_size=batch_size,
        verbose=1      # progress bar
    )
    elapsed = time.time() - start

    print(f"{name} Loss     : {loss:.6f}")
    print(f"{name} Accuracy : {acc:.6f}")
    print(f"{name} Time     : {elapsed:.2f} sec")

    return loss, acc

# -------------------------------------------------
# Run evaluations
# -------------------------------------------------
train_loss, train_acc = eval_split("Train", X_train, y_train)
val_loss, val_acc     = eval_split("Validation", X_val, y_val)
test_loss, test_acc   = eval_split("Test", X_test, y_test)

# -------------------------------------------------
# Final summary
# -------------------------------------------------
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Train Accuracy      : {train_acc:.6f}")
print(f"Validation Accuracy : {val_acc:.6f}")
print(f"Test Accuracy       : {test_acc:.6f}")




Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        36,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 412)            │       211,356 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 412)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 412)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 332)            │       137,116 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 332)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 332)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 268)            │        89,244 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 268)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 268)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 69)             │        18,561 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,477,889 (5.64 MB)

 Trainable params: 492,629 (1.88 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 985,260 (3.76 MB)


Evaluating Train
185/185 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.8592 - loss: 0.4134
Train Loss     : 0.410555
Train Accuracy : 0.860045
Train Time     : 5.82 sec

Evaluating Validation
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.8352 - loss: 0.5049
Validation Loss     : 0.496646
Validation Accuracy : 0.837984
Validation Time     : 1.58 sec

Evaluating Test
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.8410 - loss: 0.4855
Test Loss     : 0.489126
Test Accuracy : 0.840054
Test Time     : 2.56 sec

FINAL SUMMARY
Train Accuracy      : 0.860045
Validation Accuracy : 0.837984
Test Accuracy       : 0.840054


# Ensemble Model Accuracy Computation

In [4]:
#############################################
# Weighted Average (Soft Voting Ensemble)
#############################################

import numpy as np
import xgboost as xgb
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model


print("Data loaded!")
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


# ============================================================
# Batched XGBoost prediction
# ============================================================

def predict_xgb_in_batches(model, X, batch_size=4096, desc=""):
    preds = []
    for i in tqdm(range(0, len(X), batch_size), desc=desc):
        batch = X[i:i + batch_size]
        dmatrix = xgb.DMatrix(batch)
        preds.append(model.predict(dmatrix))
    return np.vstack(preds)

# ============================================================
# Predict probabilities
# ============================================================

print("Predicting XGBoost probabilities...")
prob_xgb = predict_xgb_in_batches(
    xgboost_model, X_test, batch_size=4096, desc="XGBoost TEST"
)

print("Predicting ANN probabilities...")
prob_mlp = mlp_model.predict(
    X_test, batch_size=4096, verbose=1
)

# ============================================================
# Weighted Soft Voting
# ============================================================

# Weights based on validation performance
w_xgb = 0.65
w_ann = 0.35

prob_ensemble = (w_xgb * prob_xgb) + (w_ann * prob_mlp)

y_pred_ensemble = np.argmax(prob_ensemble, axis=1)

# ============================================================
# Evaluation
# ============================================================

acc = accuracy_score(y_test, y_pred_ensemble)
print("\n===== Ensemble Accuracy =====")
print(f"{acc:.6f}")


Data loaded!
Train: (753687, 70)
Val: (188370, 70)
Test: (314088, 70)
Predicting XGBoost probabilities...


XGBoost TEST: 100%|██████████| 77/77 [00:49<00:00,  1.55it/s]


Predicting ANN probabilities...
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step

===== Ensemble Accuracy =====
0.845028
